# PDF Compressor — Backend

1. Rode a célula abaixo para instalar dependências e iniciar o servidor.
2. Copie a URL do ngrok que aparece na saída.
3. No site [PDF Compressor](https://xangrybadger.github.io/pdf-compressor/), clique em **Sem API** no header e cole a URL.

A URL muda a cada sessão — você precisa colar novamente se reiniciar o notebook.

In [ ]:
!pip install -q fastapi uvicorn pypdf Pillow python-multipart img2pdf pyngrok

In [ ]:
import os
os.makedirs('app', exist_ok=True)

%%writefile app/main.py
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pypdf import PdfReader, PdfWriter
import io
import os
from pathlib import Path

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

TEMP_DIR = Path("/tmp/pdf-compressor")
TEMP_DIR.mkdir(exist_ok=True)

@app.post("/api/compress")
async def compress_pdf(file: UploadFile = File(...)):
    try:
        content = await file.read()
        input_stream = io.BytesIO(content)
        reader = PdfReader(input_stream)
        writer = PdfWriter()
        for page in reader.pages:
            page.compress_content_streams()
            writer.add_page(page)
        output_stream = io.BytesIO()
        writer.write(output_stream)
        output_stream.seek(0)
        return FileResponse(
            output_stream,
            media_type="application/pdf",
            filename=f"compressed_{file.filename}"
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/")
async def root():
    return {"message": "PDF Compressor API"}

In [ ]:
# Coloque seu token do ngrok aqui (grátis em https://ngrok.com/signup)
NGROK_TOKEN = ''  # ← Cole entre as aspas

from pyngrok import ngrok
import threading
import uvicorn
import time

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

def run():
    uvicorn.run('app.main:app', host='0.0.0.0', port=8002)

thread = threading.Thread(target=run)
thread.daemon = True
thread.start()

time.sleep(2)

tunnel = ngrok.connect(8002)
public_url = tunnel.public_url
print(f'\n{"="*60}')
print(f'  PDF Compressor Backend Online!')
print(f'  URL: {public_url}')
print(f'{"="*60}')
print(f'\n  Cole esta URL no header do site:')
print(f'  https://xangrybadger.github.io/pdf-compressor/')
print()